In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, kruskal

# 1. Absolute Path Initializer
BASE_DIR = r"C:\Users\Admin\IPL-DataAnalysis\IPL-Data-Analysis"
MATCHES_PATH = os.path.join(BASE_DIR, "outputs", "cleaned_data", "cleaned_matches.csv")
DELIVERIES_PATH = os.path.join(BASE_DIR, "outputs", "cleaned_data", "cleaned_deliveries.csv")

matches = pd.read_csv(MATCHES_PATH)

# FIXED FALLBACK ENGINE: Verifies the file physically exists and contains data
if os.path.exists(DELIVERIES_PATH) and os.path.getsize(DELIVERIES_PATH) > 0:
    deliveries = pd.read_csv(DELIVERIES_PATH)
else:
    raw_deliveries_path = os.path.join(BASE_DIR, "data", "deliveries.csv")
    print(f"⚠️ 'outputs/cleaned_data/cleaned_deliveries.csv' was empty or missing.")
    print(f"👉 Safely falling back to raw data stream: {raw_deliveries_path}")
    deliveries = pd.read_csv(raw_deliveries_path)

# Team Standardizations
team_mappings = {
    'Delhi Daredevils': 'Delhi Capitals', 
    'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad'
}
for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(team_mappings)

print("="*60)
print("⚙️ CRICKET DATA SCIENCE LABORATORY: STATISTICAL INFERENCE ENGINE")
print("="*60)

# --- TEST 1: CHI-SQUARE TEST OF INDEPENDENCE (The Toss Myth) ---
# Question: Does winning the toss grant a statistically significant match victory advantage?
matches['won_toss_and_match'] = np.where(matches['toss_winner'] == matches['winner'], 'Won Match', 'Lost Match')
toss_contingency = pd.crosstab(matches['toss_decision'], matches['won_toss_and_match'])

print("\n[TEST 1] Contingency Table Analysis (Toss Choice vs Match Outcome):")
print(toss_contingency)

chi2, p_value, dof, expected_frequencies = chi2_contingency(toss_contingency)

print(f"\n📊 Test Results:")
print(f"-> Chi-Square Statistic: {chi2:.4f}")
print(f"-> P-Value: {p_value:.5f}")

ALPHA = 0.05
if p_value < ALPHA:
    print("\n🔥 DECISION: Reject the Null Hypothesis (H0).")
    print("Inference: The toss decision has a statistically significant relationship with winning the match.")
else:
    print("\n🛑 DECISION: Fail to Reject the Null Hypothesis (H0).")
    print("Inference: Toss advantage is a myth in this context. Match outcomes are statistically independent of toss wins.")


# --- TEST 2: KRUSKAL-WALLIS TEST (Venue Variance Validation) ---
# Question: Are score variations between major venues driven by actual ground characteristics, or are they random noise?
match_scores = deliveries.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()
first_innings = match_scores[match_scores['inning'] == 1]
venue_mapped = pd.merge(first_innings, matches[['id', 'venue']], left_on='match_id', right_on='id')

top_5_venues = venue_mapped['venue'].value_counts().nlargest(5).index
venue_groups = [venue_mapped[venue_mapped['venue'] == v]['total_runs'].values for v in top_5_venues]

print("\n" + "="*60)
print("[TEST 2] Kruskal-Wallis Test (Venue Score Distribution Parity)")
print(f"Evaluating top 5 active stadiums: {list(top_5_venues)}")

stat, p_anova = kruskal(*venue_groups)

print(f"\n📊 Test Results:")
print(f"-> H-Statistic: {stat:.4f}")
print(f"-> P-Value: {p_anova:.5e}")

if p_anova < ALPHA:
    print("\n🔥 DECISION: Reject the Null Hypothesis (H0).")
    print("Inference: Score distribution variations across these venues are highly statistically significant.")
else:
    print("\n🛑 DECISION: Fail to Reject the Null Hypothesis (H0).")
    print("Inference: Scoring differences between stadiums are not structurally distinct.")
print("="*60)

⚠️ 'outputs/cleaned_data/cleaned_deliveries.csv' was empty or missing.
👉 Safely falling back to raw data stream: C:\Users\Admin\IPL-DataAnalysis\IPL-Data-Analysis\data\deliveries.csv
⚙️ CRICKET DATA SCIENCE LABORATORY: STATISTICAL INFERENCE ENGINE

[TEST 1] Contingency Table Analysis (Toss Choice vs Match Outcome):
won_toss_and_match  Lost Match  Won Match
toss_decision                            
bat                        213        177
field                      328        372

📊 Test Results:
-> Chi-Square Statistic: 5.7240
-> P-Value: 0.01673

🔥 DECISION: Reject the Null Hypothesis (H0).
Inference: The toss decision has a statistically significant relationship with winning the match.

[TEST 2] Kruskal-Wallis Test (Venue Score Distribution Parity)
Evaluating top 5 active stadiums: ['Eden Gardens', 'Wankhede Stadium', 'M Chinnaswamy Stadium', 'Feroz Shah Kotla', 'Rajiv Gandhi International Stadium, Uppal']

📊 Test Results:
-> H-Statistic: 5.3699
-> P-Value: 2.51409e-01

🛑 DECISION: 